In [ ]:
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from datetime import datetime

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
# =========================================================
# 1. PREPARE DAILY PROFILES ONCE
# =========================================================

def prepare_daily_profiles(df_input, feature_cols,target_col,samples_per_day=48):
    """
    Validate and reshape half-hourly data into daily profiles.

    Steps:
    - keep only days with exactly `samples_per_day` rows
    - keep only days with no missing values in feature_cols + target_col
    - reshape to daily wide format

    Parameters
    ----------
    df_input : pandas.DataFrame
        Half-hourly dataframe with a DatetimeIndex.
    feature_cols : list of str
        Weather/input columns used as clustering features.
    target_col : str
        Power/output column.
    samples_per_day : int, default=48
        Expected number of half-hour samples per full day.

    Returns
    -------
    X_daily : pandas.DataFrame
        One row per valid day, containing all daily weather values.
    y_daily : pandas.DataFrame
        One row per valid day, containing the full-day target profile.
    valid_days : np.ndarray
        Sorted array of valid day labels.
    """
    required_cols = feature_cols + [target_col]

    df_work = df_input.copy().sort_index()
    df_work["date"] = df_work.index.date
    df_work["time"] = df_work.index.strftime("%H:%M")

    # Keep only full days
    rows_per_day = df_work.groupby("date").size()
    full_days = rows_per_day[rows_per_day == samples_per_day].index
    df_work = df_work[df_work["date"].isin(full_days)].copy()

    # Keep only days with complete data in required columns
    valid_day_mask = (
        df_work.groupby("date")[required_cols]
        .apply(lambda day_data: not day_data.isna().any().any())
    )
    valid_days = np.array(sorted(valid_day_mask[valid_day_mask].index))

    df_work = df_work[df_work["date"].isin(valid_days)].copy()

    # Build daily weather feature matrix
    daily_feature_blocks = []

    for feature_name in feature_cols:
        daily_feature_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=feature_name
        )
        daily_feature_matrix = daily_feature_matrix.reindex(
            sorted(daily_feature_matrix.columns),
            axis=1
        )
        daily_feature_matrix.columns = [
            f"{feature_name}_{time_label}"
            for time_label in daily_feature_matrix.columns
        ]
        daily_feature_blocks.append(daily_feature_matrix)

    X_daily = pd.concat(daily_feature_blocks, axis=1)

    # Build daily target matrix
    daily_target_matrix = df_work.pivot(
        index="date",
        columns="time",
        values=target_col
    )
    daily_target_matrix = daily_target_matrix.reindex(
        sorted(daily_target_matrix.columns),
        axis=1
    )
    daily_target_matrix.columns = [
        f"{target_col}_{time_label}"
        for time_label in daily_target_matrix.columns
    ]

    # Final alignment
    common_days = X_daily.index.intersection(daily_target_matrix.index)
    X_daily = X_daily.loc[common_days]
    y_daily = daily_target_matrix.loc[common_days]
    valid_days = np.array(sorted(common_days))

    return X_daily, y_daily, valid_days

**CALL ON FUNCTIONS**

In [ ]:
final_file = "data/complete_dataframe_30min.csv"
df = pd.read_csv(str(ROOT/final_file), delimiter=',', decimal='.', parse_dates=['ts'], index_col='ts') #ts is already the index :)



In [ ]:
# =========================================================
# 2. CREATE MONTHLY SAMPLED-DAY FOLDS FROM valid_days
# =========================================================

def make_monthly_sampled_folds_from_days(
    valid_days,
    n_folds=5,
    test_days_per_month=3,
    random_state=42
):
    """
    Create folds from an existing set of valid days.

    Each fold:
    - samples `test_days_per_month` days from each month as test days
    - uses all remaining valid days as train days

    Parameters
    ----------
    valid_days : array-like
        Array of valid dates.
    n_folds : int, default=5
        Number of folds.
    test_days_per_month : int, default=3
        Number of test days to sample from each month in each fold.
    random_state : int, default=42
        Seed for reproducibility.

    Returns
    -------
    folds : list of dict
        Each fold has:
        {
            "fold_id": int,
            "train_days": np.ndarray,
            "test_days": np.ndarray
        }
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    folds = []

    for fold_id in range(n_folds):
        sampled_test_days = []

        for _, month_group in valid_day_table.groupby("month"):
            month_days = month_group["date"].dt.date.values
            n_test_days = min(test_days_per_month, len(month_days))

            chosen_test_days = rng.choice(
                month_days,
                size=n_test_days,
                replace=False
            )
            sampled_test_days.extend(chosen_test_days.tolist())

        test_days = np.array(sorted(set(sampled_test_days)))
        train_days = np.array(sorted(set(valid_days) - set(test_days)))

        folds.append({
            "fold_id": fold_id,
            "train_days": train_days,
            "test_days": test_days
        })

    return folds


**CALL ON FUNCTIONS**

In [ ]:
# =========================================================
# 3. EVALUATE ONE K USING ALREADY PREPARED DAILY MATRICES
# =========================================================

def evaluate_kmeans_for_k_daily_matrices(
    X_train_daily,
    y_train_daily,
    X_test_daily,
    y_test_daily,
    k,
    random_state=42
):
    """
    Train/evaluate one KMeans model for one K using already prepared daily matrices.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_daily)
    X_test_scaled = scaler.transform(X_test_daily)

    kmeans_model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    train_cluster_labels = kmeans_model.fit_predict(X_train_scaled)
    test_cluster_labels = kmeans_model.predict(X_test_scaled)

    # Learn mean full-day power profile from TRAIN only
    train_targets_with_clusters = y_train_daily.copy()
    train_targets_with_clusters["cluster"] = train_cluster_labels
    cluster_power_profiles = train_targets_with_clusters.groupby("cluster").mean()

    # Predict each TEST day's full-day power profile
    y_pred_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )

    for day, cluster_label in zip(y_test_daily.index, test_cluster_labels):
        y_pred_daily.loc[day] = cluster_power_profiles.loc[cluster_label].values

    # Flatten all day-time points for global metrics
    y_true_flat = y_test_daily.to_numpy().flatten()
    y_pred_flat = y_pred_daily.to_numpy().flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))

    # Average per-day profile MAE
    daily_mae = np.mean(
        np.abs(y_test_daily.to_numpy() - y_pred_daily.to_numpy()).mean(axis=1)
    )

    return {
        "k": k,
        "scaler": scaler,
        "kmeans_model": kmeans_model,
        "train_cluster_labels": train_cluster_labels,
        "test_cluster_labels": test_cluster_labels,
        "cluster_power_profiles": cluster_power_profiles,
        "y_pred_daily": y_pred_daily,
        "mae": mae,
        "rmse": rmse,
        "daily_mae": daily_mae
    }

**CALL ON FUNCTIONS**

In [ ]:
# =========================================================
# 4. EVALUATE ONE K ACROSS FIXED FOLDS
# =========================================================

def evaluate_k_across_folds(
    X_daily,
    y_daily,
    folds,
    k,
    random_state=42
):
    """
    Evaluate one candidate K across a set of fixed folds.
    """
    fold_metrics = []

    for fold_definition in folds:
        train_days = fold_definition["train_days"]
        test_days = fold_definition["test_days"]

        X_train_daily = X_daily.loc[train_days]
        y_train_daily = y_daily.loc[train_days]

        X_test_daily = X_daily.loc[test_days]
        y_test_daily = y_daily.loc[test_days]

        fold_result = evaluate_kmeans_for_k_daily_matrices(
            X_train_daily=X_train_daily,
            y_train_daily=y_train_daily,
            X_test_daily=X_test_daily,
            y_test_daily=y_test_daily,
            k=k,
            random_state=random_state
        )

        fold_metrics.append({
            "k": k,
            "fold_id": fold_definition["fold_id"],
            "n_train_days": len(X_train_daily),
            "n_test_days": len(X_test_daily),
            "mae": fold_result["mae"],
            "rmse": fold_result["rmse"],
            "daily_mae": fold_result["daily_mae"]
        })

    fold_results_df = pd.DataFrame(fold_metrics)

    summary = {
        "k": k,
        "mean_mae": fold_results_df["mae"].mean(),
        "std_mae": fold_results_df["mae"].std(ddof=1),
        "mean_rmse": fold_results_df["rmse"].mean(),
        "std_rmse": fold_results_df["rmse"].std(ddof=1),
        "mean_daily_mae": fold_results_df["daily_mae"].mean(),
        "std_daily_mae": fold_results_df["daily_mae"].std(ddof=1)
    }

    return fold_results_df, summary

**CALL ON FUNCTIONS**

In [ ]:
# =========================================================
# 5. SEARCH BEST K USING THE SAME FOLDS
# =========================================================

def search_best_k_with_folds(
    X_daily,
    y_daily,
    valid_days,
    k_values,
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42
):
    """
    Generate folds once from valid_days, reuse them across all K values,
    and choose the best K from average fold performance.
    """
    folds = make_monthly_sampled_folds_from_days(
        valid_days=valid_days,
        n_folds=n_folds,
        test_days_per_month=test_days_per_month,
        random_state=fold_random_state
    )

    all_fold_results = []
    k_summary_rows = []

    for k in k_values:
        fold_results_df, k_summary = evaluate_k_across_folds(
            X_daily=X_daily,
            y_daily=y_daily,
            folds=folds,
            k=k,
            random_state=model_random_state
        )

        all_fold_results.append(fold_results_df)
        k_summary_rows.append(k_summary)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    summary_df = pd.DataFrame(k_summary_rows).sort_values("k").reset_index(drop=True)

    best_k_row = summary_df.loc[summary_df["mean_daily_mae"].idxmin()]
    best_k = int(best_k_row["k"])

    return {
        "folds": folds,
        "all_fold_results": all_fold_results_df,
        "summary": summary_df,
        "best_k": best_k
    }

**CALL ON FUNCTIONS**

In [ ]:
# =========================================================
# 6. CREATE A FRESH FINAL MONTHLY-BALANCED SPLIT
# =========================================================

def create_final_monthly_split_from_days(
    valid_days,
    test_days_per_month=3,
    random_state=999
):
    """
    Create one fresh final split from valid_days:
    - final test = selected number of days per month
    - final train = all remaining valid days
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    sampled_final_test_days = []

    for _, month_group in valid_day_table.groupby("month"):
        month_days = month_group["date"].dt.date.values
        n_test_days = min(test_days_per_month, len(month_days))

        chosen_test_days = rng.choice(
            month_days,
            size=n_test_days,
            replace=False
        )
        sampled_final_test_days.extend(chosen_test_days.tolist())

    final_test_days = np.array(sorted(set(sampled_final_test_days)))
    final_train_days = np.array(sorted(set(valid_days) - set(final_test_days)))

    return {
        "train_days": final_train_days,
        "test_days": final_test_days
    }


**CALL ON FUNCTIONS**

In [ ]:
# =========================================================
# 7. FINAL MODEL FIT WITH CHOSEN K
# =========================================================

def fit_final_model(
    X_daily,
    y_daily,
    valid_days,
    best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
):
    """
    After best K is selected:
    - create a fresh final split
    - retrain/evaluate with chosen K
    """
    final_split = create_final_monthly_split_from_days(
        valid_days=valid_days,
        test_days_per_month=final_test_days_per_month,
        random_state=final_split_random_state
    )

    final_train_days = final_split["train_days"]
    final_test_days = final_split["test_days"]

    X_train_daily = X_daily.loc[final_train_days]
    y_train_daily = y_daily.loc[final_train_days]

    X_test_daily = X_daily.loc[final_test_days]
    y_test_daily = y_daily.loc[final_test_days]

    final_model_result = evaluate_kmeans_for_k_daily_matrices(
        X_train_daily=X_train_daily,
        y_train_daily=y_train_daily,
        X_test_daily=X_test_daily,
        y_test_daily=y_test_daily,
        k=best_k,
        random_state=model_random_state
    )

    return {
        "best_k": best_k,
        "final_train_days": final_train_days,
        "final_test_days": final_test_days,
        "X_train_daily": X_train_daily,
        "y_train_daily": y_train_daily,
        "X_test_daily": X_test_daily,
        "y_test_daily": y_test_daily,
        "model_result": final_model_result
    }


In [ ]:
# =========================================================
# OPTIONAL: EXPORT / INSPECT FOLDS
# =========================================================

def folds_to_dataframe(folds):
    """
    Convert folds into a long-format dataframe.
    Each day appears once per fold with role=train/test.
    """
    fold_rows = []

    for fold_definition in folds:
        fold_id = fold_definition["fold_id"]

        for train_day in fold_definition["train_days"]:
            fold_rows.append({
                "date": pd.to_datetime(train_day),
                "fold_id": fold_id,
                "role": "train"
            })

        for test_day in fold_definition["test_days"]:
            fold_rows.append({
                "date": pd.to_datetime(test_day),
                "fold_id": fold_id,
                "role": "test"
            })

    fold_df = pd.DataFrame(fold_rows).sort_values(["fold_id", "date"]).reset_index(drop=True)
    return fold_df


def save_folds_to_csv(folds, filepath):
    """
    Optional helper to save folds to CSV.
    """
    fold_df = folds_to_dataframe(folds)
    fold_df.to_csv(filepath, index=False)


def load_folds_from_csv(filepath):
    """
    Optional helper to reconstruct folds from a CSV file.
    """
    fold_df = pd.read_csv(filepath, parse_dates=["date"])

    folds = []

    for fold_id, fold_group in fold_df.groupby("fold_id"):
        train_days = fold_group.loc[
            fold_group["role"] == "train", "date"
        ].dt.date.values

        test_days = fold_group.loc[
            fold_group["role"] == "test", "date"
        ].dt.date.values

        folds.append({
            "fold_id": int(fold_id),
            "train_days": np.array(sorted(train_days)),
            "test_days": np.array(sorted(test_days))
        })

    return folds

UPDATED WITH SOLAR AND WIND OUTPUTS!

NOTE: One modeling caution

This works well if you truly want:

one common day classification based on weather
then separate PV and wind outputs from that same classification

But if PV and wind respond to very different subsets of the weather variables, sometimes the best clustering for PV is not the best clustering for wind.

So this shared-cluster approach is great if your goal is:

one unified weather-day typology
two output profiles from it

That sounds like what you want.

In [ ]:
#FULL CODE VERSION

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error


# =========================================================
# 0.5 AGGREGATE POWER COLUMNS
# =========================================================

def aggregate_power_columns(
    df_input,
    pv_cols=None,
    wind_power_cols=None,
    drop_original=False
):
    """
    Aggregate multiple PV and wind power columns into total signals.
    """
    df_output = df_input.copy()

    if pv_cols is not None:
        missing_pv = [col for col in pv_cols if col not in df_output.columns]
        if missing_pv:
            raise ValueError(f"Missing PV columns: {missing_pv}")

        df_output["PV_total"] = df_output[pv_cols].sum(axis=1, min_count=1)

    if wind_power_cols is not None:
        missing_wind = [col for col in wind_power_cols if col not in df_output.columns]
        if missing_wind:
            raise ValueError(f"Missing wind columns: {missing_wind}")

        df_output["Wind_total"] = df_output[wind_power_cols].sum(axis=1, min_count=1)

    if drop_original:
        cols_to_drop = []
        if pv_cols is not None:
            cols_to_drop.extend(pv_cols)
        if wind_power_cols is not None:
            cols_to_drop.extend(wind_power_cols)

        df_output = df_output.drop(columns=cols_to_drop)

    return df_output


# =========================================================
# 1. PREPARE DAILY PROFILES ONCE
# =========================================================

def prepare_daily_profiles_multi_output(
    df_input,
    feature_cols,
    target_cols,
    samples_per_day=48
):
    """
    Validate and reshape half-hourly data into daily profiles.

    Returns
    -------
    X_daily : DataFrame
        One row per valid day, containing all daily weather values.
    y_daily_dict : dict[str, DataFrame]
        Dictionary of daily target matrices, one per target column.
    valid_days : np.ndarray
        Sorted array of valid day labels.
    """
    required_cols = feature_cols + target_cols

    df_work = df_input.copy().sort_index()
    df_work["date"] = df_work.index.date
    df_work["time"] = df_work.index.strftime("%H:%M")

    # Keep only full days
    rows_per_day = df_work.groupby("date").size()
    full_days = rows_per_day[rows_per_day == samples_per_day].index
    df_work = df_work[df_work["date"].isin(full_days)].copy()

    # Keep only days with complete data in required columns
    valid_day_mask = (
        df_work.groupby("date")[required_cols]
        .apply(lambda day_data: not day_data.isna().any().any())
    )
    valid_days = np.array(sorted(valid_day_mask[valid_day_mask].index))
    df_work = df_work[df_work["date"].isin(valid_days)].copy()

    # Build daily weather feature matrix
    daily_feature_blocks = []

    for feature_name in feature_cols:
        daily_feature_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=feature_name
        )
        daily_feature_matrix = daily_feature_matrix.reindex(
            sorted(daily_feature_matrix.columns),
            axis=1
        )
        daily_feature_matrix.columns = [
            f"{feature_name}_{time_label}"
            for time_label in daily_feature_matrix.columns
        ]
        daily_feature_blocks.append(daily_feature_matrix)

    X_daily = pd.concat(daily_feature_blocks, axis=1)

    # Build one daily target matrix per target column
    y_daily_dict = {}

    for target_col in target_cols:
        daily_target_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=target_col
        )
        daily_target_matrix = daily_target_matrix.reindex(
            sorted(daily_target_matrix.columns),
            axis=1
        )
        daily_target_matrix.columns = [
            f"{target_col}_{time_label}"
            for time_label in daily_target_matrix.columns
        ]
        y_daily_dict[target_col] = daily_target_matrix

    # Align everything on common days
    common_days = X_daily.index.copy()
    for target_col in target_cols:
        common_days = common_days.intersection(y_daily_dict[target_col].index)

    X_daily = X_daily.loc[common_days]
    for target_col in target_cols:
        y_daily_dict[target_col] = y_daily_dict[target_col].loc[common_days]

    valid_days = np.array(sorted(common_days))

    return X_daily, y_daily_dict, valid_days


# =========================================================
# 2. CREATE MONTHLY SAMPLED-DAY FOLDS FROM valid_days
# =========================================================

def make_monthly_sampled_folds_from_days(
    valid_days,
    n_folds=5,
    test_days_per_month=3,
    random_state=42
):
    """
    Create folds from an existing set of valid days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    folds = []

    for fold_id in range(n_folds):
        sampled_test_days = []

        for _, month_group in valid_day_table.groupby("month"):
            month_days = month_group["date"].dt.date.values
            n_test_days = min(test_days_per_month, len(month_days))

            chosen_test_days = rng.choice(
                month_days,
                size=n_test_days,
                replace=False
            )
            sampled_test_days.extend(chosen_test_days.tolist())

        test_days = np.array(sorted(set(sampled_test_days)))
        train_days = np.array(sorted(set(valid_days) - set(test_days)))

        folds.append({
            "fold_id": fold_id,
            "train_days": train_days,
            "test_days": test_days
        })

    return folds


# =========================================================
# 3. METRIC HELPER
# =========================================================

def compute_profile_metrics(y_true_daily, y_pred_daily):
    """
    Compute global and per-day profile metrics for one target.
    """
    y_true_flat = y_true_daily.to_numpy().flatten()
    y_pred_flat = y_pred_daily.to_numpy().flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    daily_mae = np.mean(
        np.abs(y_true_daily.to_numpy() - y_pred_daily.to_numpy()).mean(axis=1)
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "daily_mae": daily_mae
    }


# =========================================================
# 4. EVALUATE ONE K WITH SAME WEATHER CLUSTERS, TWO OUTPUTS
# =========================================================

def evaluate_kmeans_for_k_multi_output(
    X_train_daily,
    y_train_dict,
    X_test_daily,
    y_test_dict,
    k,
    random_state=42
):
    """
    Train one KMeans model on weather-day profiles and predict multiple
    daily power targets from the same cluster assignments.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_daily)
    X_test_scaled = scaler.transform(X_test_daily)

    kmeans_model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    train_cluster_labels = kmeans_model.fit_predict(X_train_scaled)
    test_cluster_labels = kmeans_model.predict(X_test_scaled)

    cluster_profile_dict = {}
    y_pred_dict = {}
    metrics_dict = {}

    for target_name, y_train_daily in y_train_dict.items():
        y_test_daily = y_test_dict[target_name]

        train_targets_with_clusters = y_train_daily.copy()
        train_targets_with_clusters["cluster"] = train_cluster_labels

        cluster_profiles = train_targets_with_clusters.groupby("cluster").mean()
        cluster_profile_dict[target_name] = cluster_profiles

        y_pred_daily = pd.DataFrame(
            index=y_test_daily.index,
            columns=y_test_daily.columns,
            dtype=float
        )

        for day, cluster_label in zip(y_test_daily.index, test_cluster_labels):
            y_pred_daily.loc[day] = cluster_profiles.loc[cluster_label].values

        y_pred_dict[target_name] = y_pred_daily
        metrics_dict[target_name] = compute_profile_metrics(y_test_daily, y_pred_daily)

    return {
        "k": k,
        "scaler": scaler,
        "kmeans_model": kmeans_model,
        "train_cluster_labels": train_cluster_labels,
        "test_cluster_labels": test_cluster_labels,
        "cluster_profile_dict": cluster_profile_dict,
        "y_pred_dict": y_pred_dict,
        "metrics_dict": metrics_dict
    }


# =========================================================
# 5. EVALUATE ONE K ACROSS FIXED FOLDS
# =========================================================

def evaluate_k_across_folds_multi_output(
    X_daily,
    y_daily_dict,
    folds,
    k,
    random_state=42
):
    """
    Evaluate one candidate K across fixed folds for multiple output targets.
    """
    fold_metrics = []

    target_names = list(y_daily_dict.keys())

    for fold_definition in folds:
        train_days = fold_definition["train_days"]
        test_days = fold_definition["test_days"]

        X_train_daily = X_daily.loc[train_days]
        X_test_daily = X_daily.loc[test_days]

        y_train_dict = {
            target_name: y_daily_dict[target_name].loc[train_days]
            for target_name in target_names
        }
        y_test_dict = {
            target_name: y_daily_dict[target_name].loc[test_days]
            for target_name in target_names
        }

        fold_result = evaluate_kmeans_for_k_multi_output(
            X_train_daily=X_train_daily,
            y_train_dict=y_train_dict,
            X_test_daily=X_test_daily,
            y_test_dict=y_test_dict,
            k=k,
            random_state=random_state
        )

        row = {
            "k": k,
            "fold_id": fold_definition["fold_id"],
            "n_train_days": len(train_days),
            "n_test_days": len(test_days)
        }

        for target_name in target_names:
            target_metrics = fold_result["metrics_dict"][target_name]
            row[f"{target_name}_mae"] = target_metrics["mae"]
            row[f"{target_name}_rmse"] = target_metrics["rmse"]
            row[f"{target_name}_daily_mae"] = target_metrics["daily_mae"]

        fold_metrics.append(row)

    fold_results_df = pd.DataFrame(fold_metrics)

    summary = {"k": k}

    for target_name in target_names:
        summary[f"mean_{target_name}_mae"] = fold_results_df[f"{target_name}_mae"].mean()
        summary[f"std_{target_name}_mae"] = fold_results_df[f"{target_name}_mae"].std(ddof=1)
        summary[f"mean_{target_name}_rmse"] = fold_results_df[f"{target_name}_rmse"].mean()
        summary[f"std_{target_name}_rmse"] = fold_results_df[f"{target_name}_rmse"].std(ddof=1)
        summary[f"mean_{target_name}_daily_mae"] = fold_results_df[f"{target_name}_daily_mae"].mean()
        summary[f"std_{target_name}_daily_mae"] = fold_results_df[f"{target_name}_daily_mae"].std(ddof=1)

    # Combined score for choosing K: average of target daily MAEs
    summary["mean_combined_daily_mae"] = np.mean(
        [summary[f"mean_{target_name}_daily_mae"] for target_name in target_names]
    )

    return fold_results_df, summary


# =========================================================
# 6. SEARCH BEST K USING SAME WEATHER CLUSTERS FOR BOTH TARGETS
# =========================================================

def search_best_k_with_folds_multi_output(
    X_daily,
    y_daily_dict,
    valid_days,
    k_values,
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42
):
    """
    Generate folds once from valid_days, reuse them across all K values,
    and choose best K from combined average fold performance.
    """
    folds = make_monthly_sampled_folds_from_days(
        valid_days=valid_days,
        n_folds=n_folds,
        test_days_per_month=test_days_per_month,
        random_state=fold_random_state
    )

    all_fold_results = []
    k_summary_rows = []

    for k in k_values:
        fold_results_df, k_summary = evaluate_k_across_folds_multi_output(
            X_daily=X_daily,
            y_daily_dict=y_daily_dict,
            folds=folds,
            k=k,
            random_state=model_random_state
        )

        all_fold_results.append(fold_results_df)
        k_summary_rows.append(k_summary)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    summary_df = pd.DataFrame(k_summary_rows).sort_values("k").reset_index(drop=True)

    best_k_row = summary_df.loc[summary_df["mean_combined_daily_mae"].idxmin()]
    best_k = int(best_k_row["k"])

    return {
        "folds": folds,
        "all_fold_results": all_fold_results_df,
        "summary": summary_df,
        "best_k": best_k
    }


# =========================================================
# 7. CREATE A FRESH FINAL MONTHLY-BALANCED SPLIT
# =========================================================

def create_final_monthly_split_from_days(
    valid_days,
    test_days_per_month=3,
    random_state=999
):
    """
    Create one fresh final split from valid_days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    sampled_final_test_days = []

    for _, month_group in valid_day_table.groupby("month"):
        month_days = month_group["date"].dt.date.values
        n_test_days = min(test_days_per_month, len(month_days))

        chosen_test_days = rng.choice(
            month_days,
            size=n_test_days,
            replace=False
        )
        sampled_final_test_days.extend(chosen_test_days.tolist())

    final_test_days = np.array(sorted(set(sampled_final_test_days)))
    final_train_days = np.array(sorted(set(valid_days) - set(final_test_days)))

    return {
        "train_days": final_train_days,
        "test_days": final_test_days
    }


# =========================================================
# 8. FINAL MODEL FIT
# =========================================================

def fit_final_model_multi_output(
    X_daily,
    y_daily_dict,
    valid_days,
    best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
):
    """
    After best K is selected:
    - create a fresh final split
    - retrain/evaluate with chosen K
    """
    final_split = create_final_monthly_split_from_days(
        valid_days=valid_days,
        test_days_per_month=final_test_days_per_month,
        random_state=final_split_random_state
    )

    final_train_days = final_split["train_days"]
    final_test_days = final_split["test_days"]

    X_train_daily = X_daily.loc[final_train_days]
    X_test_daily = X_daily.loc[final_test_days]

    y_train_dict = {
        target_name: target_matrix.loc[final_train_days]
        for target_name, target_matrix in y_daily_dict.items()
    }
    y_test_dict = {
        target_name: target_matrix.loc[final_test_days]
        for target_name, target_matrix in y_daily_dict.items()
    }

    final_model_result = evaluate_kmeans_for_k_multi_output(
        X_train_daily=X_train_daily,
        y_train_dict=y_train_dict,
        X_test_daily=X_test_daily,
        y_test_dict=y_test_dict,
        k=best_k,
        random_state=model_random_state
    )

    return {
        "best_k": best_k,
        "final_train_days": final_train_days,
        "final_test_days": final_test_days,
        "X_train_daily": X_train_daily,
        "X_test_daily": X_test_daily,
        "y_train_dict": y_train_dict,
        "y_test_dict": y_test_dict,
        "model_result": final_model_result
    }


# =========================================================
# OPTIONAL: EXPORT / INSPECT FOLDS
# =========================================================

def folds_to_dataframe(folds):
    fold_rows = []

    for fold_definition in folds:
        fold_id = fold_definition["fold_id"]

        for train_day in fold_definition["train_days"]:
            fold_rows.append({
                "date": pd.to_datetime(train_day),
                "fold_id": fold_id,
                "role": "train"
            })

        for test_day in fold_definition["test_days"]:
            fold_rows.append({
                "date": pd.to_datetime(test_day),
                "fold_id": fold_id,
                "role": "test"
            })

    fold_df = pd.DataFrame(fold_rows).sort_values(["fold_id", "date"]).reset_index(drop=True)
    return fold_df


def save_folds_to_csv(folds, filepath):
    fold_df = folds_to_dataframe(folds)
    fold_df.to_csv(filepath, index=False)


def load_folds_from_csv(filepath):
    fold_df = pd.read_csv(filepath, parse_dates=["date"])

    folds = []

    for fold_id, fold_group in fold_df.groupby("fold_id"):
        train_days = fold_group.loc[
            fold_group["role"] == "train", "date"
        ].dt.date.values

        test_days = fold_group.loc[
            fold_group["role"] == "test", "date"
        ].dt.date.values

        folds.append({
            "fold_id": int(fold_id),
            "train_days": np.array(sorted(train_days)),
            "test_days": np.array(sorted(test_days))
        })

    return folds

In [ ]:
#CALLING ON THE CODE GUIDELINE

# Power columns to aggregate
pv_cols = ['B117_m','B319_m','B330_1_m','B330_2_m','B330_3_m','B716_m','B715_m']
wind_power_cols = ['Aircon_WT Power_m', 'Gaia_WT Power_m']

# Step 0.5: aggregate power columns
df = aggregate_power_columns(
    df_input=df,
    pv_cols=pv_cols,
    wind_power_cols=wind_power_cols,
    drop_original=False
)

# Weather features used for clustering
feature_cols = ["temperature", "irradiance", "windspeed"]

# Two targets predicted from the same weather clusters
target_cols = ["PV_total", "Wind_total"]

# Candidate K values
k_values = [2, 3, 4, 5, 6, 7]

# Step 1: prepare daily profiles once
X_daily, y_daily_dict, valid_days = prepare_daily_profiles_multi_output(
    df_input=df,
    feature_cols=feature_cols,
    target_cols=target_cols,
    samples_per_day=48
)

print("Number of valid days:", len(valid_days))
print("X_daily shape:", X_daily.shape)
print("PV daily shape:", y_daily_dict["PV_total"].shape)
print("Wind daily shape:", y_daily_dict["Wind_total"].shape)

# Step 2: choose best K using folds
search_result = search_best_k_with_folds_multi_output(
    X_daily=X_daily,
    y_daily_dict=y_daily_dict,
    valid_days=valid_days,
    k_values=k_values,
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42
)

best_k = search_result["best_k"]
folds = search_result["folds"]

print("Best K:", best_k)
print(search_result["summary"])

# Optional: inspect fold membership
fold_df = folds_to_dataframe(folds)
print(fold_df.head())

# Step 3: fit final model with fresh final split
final_result = fit_final_model_multi_output(
    X_daily=X_daily,
    y_daily_dict=y_daily_dict,
    valid_days=valid_days,
    best_k=best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
)

print("Final PV daily MAE:", final_result["model_result"]["metrics_dict"]["PV_total"]["daily_mae"])
print("Final Wind daily MAE:", final_result["model_result"]["metrics_dict"]["Wind_total"]["daily_mae"])
print("Final PV RMSE:", final_result["model_result"]["metrics_dict"]["PV_total"]["rmse"])
print("Final Wind RMSE:", final_result["model_result"]["metrics_dict"]["Wind_total"]["rmse"])

